# PD06 · PQRS completas: agregaciones de alta cardinalidad en Polars CPU y GPU
Curso de Postgrado: BigData, Especialización en Bases de datos · CyberColombia.

Compararás **Polars CPU frente a Polars GPU**, con todos los datos de tres
periodos de Supersalud, sin pandas para las consultas. Resolverás dos ejercicios:

1. Departamentos con más quejas según edad, patología, motivo y canal.
2. Evolución de esas mismas combinaciones mes a mes, en orden cronológico ascendente.

Las claves de texto y el número de grupos aumentan el trabajo de agregación y
ordenación. Debes medirlo: no se garantiza saturar la GPU ni obtener aceleración.
No copies filas para simular más datos. Registra la cardinalidad observada.

## 1. Preparar los tres CSV completos

Desde una terminal Ubuntu/WSL, en la raíz del repositorio y con el entorno activo:

```bash
python kit/00_datos.py --descargar pqrs-completos
python kit/00_datos.py --verificar pqrs-completos
```

Deben quedar en `kit/data/pqrs_completos/`:

| Corte | CSV | Reportes de referencia |
|---|---|---:|
| 2023-II | `PQRD_2023_II.csv` | 716.697 |
| 2024-I | `PQRD_2024_I.csv` | 781.601 |
| 2024-II | `PQRD_2024_II.csv` | 946.468 |

Total: **2.444.766 reportes**, aproximadamente 1,82 GB de CSV. El descargador
conserva archivos correctos y verifica los tres. No son muestras.
No ejecutes esos comandos dentro del reloj del benchmark.


In [4]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
import json, os, platform, subprocess, sys
import polars as pl
from polars.testing import assert_frame_equal
from IPython.display import display

REPOSITORIO = None  # Ejemplo: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Indicar la ruta del repositorio'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
# Opcional: si los CSV verificados están fuera del repositorio, indicar su carpeta.
# Ejemplo WSL: Path('/mnt/c/Users/carde/OneDrive/Documentos/CSV')
CSV_DIR_ALTERNATIVO = None
CSV_DIR = Path(CSV_DIR_ALTERNATIVO).expanduser().resolve() if CSV_DIR_ALTERNATIVO else KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/soluciones/PD06'
PARQUET_DIR = OUT / 'parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PERIODOS = ['2023_II', '2024_I', '2024_II']
ARCHIVOS_CSV = [CSV_DIR / f'PQRD_{periodo}.csv' for periodo in PERIODOS]
ARCHIVOS_PARQUET = [PARQUET_DIR / f'{periodo}.parquet' for periodo in PERIODOS]
FILAS_ESPERADAS = [716697, 781601, 946468]
TOTAL_ESPERADO = sum(FILAS_ESPERADAS)
REPETICIONES = 3
TOP_N = 10
DISPOSITIVO_CUDA = 0
assert all(p.is_file() for p in ARCHIVOS_CSV), 'Completar descarga y verificación'
assert REPETICIONES >= 3
print('Python:', sys.executable)
print('Polars:', pl.__version__)
print('Entrada:', CSV_DIR)
print('Resultados:', OUT)

Python: /mnt/c/Users/TUPTC/.venv-gpu/bin/python
Polars: 1.35.2
Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_completos
Resultados: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06


## 2. Leer y convertir con Polars · desarrollo del estudiante

Completa la siguiente celda sin pandas ni funciones del taller que oculten la conversión:

1. Inspecciona cinco filas de cada CSV con `pl.read_csv`. Usa `;`, `,` y `;`,
   respectivamente. Conserva códigos y las 38 columnas originales como texto.
2. Usa `pl.scan_csv` y transforma con expresiones Polars. Añade:
   - `corte_archivo`: corte del CSV de origen.
   - `codigo_departamento`: desde `afec_cod_depto`, dos dígitos; marca inválidos.
   - `edad_anios`: desde `afec_edad`, numérica. Explicita el tratamiento de edades
     vacías, negativas, no numéricas o no finitas; conserva estos reportes en un grupo nulo.
   - `anio` y `mes_num`: números desde `periodo` y `mes`, no desde el nombre semestral.
3. Escribe un Parquet por corte con `sink_parquet`, Zstandard y streaming CPU.
4. Comprueba que están los tres, que conservan sus filas y el contrato de columnas.
5. Al reejecutar, comprueba los Parquet existentes antes de decidir reutilizarlos.

La preparación se hace con Polars CPU; la GPU se utilizará para las agregaciones.
No mezcles este tiempo con los de consulta. Guarda el tiempo de conversión aparte.


In [5]:
# 2.1 Reglas explícitas de lectura y normalización

SEPARADORES = {'2023_II': ';', '2024_I': ',', '2024_II': ';'}

# Códigos DIVIPOLA de los 32 departamentos + Bogotá D. C. Fuera de esta lista → inválido.
DEPARTAMENTOS = {
    '05': 'Antioquia', '08': 'Atlántico', '11': 'Bogotá D. C.', '13': 'Bolívar',
    '15': 'Boyacá', '17': 'Caldas', '18': 'Caquetá', '19': 'Cauca', '20': 'Cesar',
    '23': 'Córdoba', '25': 'Cundinamarca', '27': 'Chocó', '41': 'Huila',
    '44': 'La Guajira', '47': 'Magdalena', '50': 'Meta', '52': 'Nariño',
    '54': 'Norte de Santander', '63': 'Quindío', '66': 'Risaralda', '68': 'Santander',
    '70': 'Sucre', '73': 'Tolima', '76': 'Valle del Cauca', '81': 'Arauca',
    '85': 'Casanare', '86': 'Putumayo', '88': 'San Andrés y Providencia',
    '91': 'Amazonas', '94': 'Guainía', '95': 'Guaviare', '97': 'Vaupés', '99': 'Vichada'}

# Edad: texto → Float64 (no numérica → nulo). Se conserva solo si es finita, >= 0 y
# <= EDAD_MAXIMA; se trunca a años cumplidos (floor) y se guarda como Int16.
# Vacías, negativas, no numéricas, NaN/inf o > EDAD_MAXIMA → edad_anios nula
# (el reporte NO se elimina: queda en el grupo de edad nula).
EDAD_MAXIMA = 120

# Fecha: anio desde `periodo` y mes_num desde `mes`; se aceptan números o nombres en español.
# Si año o mes no son válidos, ambos quedan nulos y fecha_valida = False.
ANIO_MINIMO, ANIO_MAXIMO = 2000, 2100
MESES_TEXTO = {'enero': '1', 'febrero': '2', 'marzo': '3', 'abril': '4', 'mayo': '5',
               'junio': '6', 'julio': '7', 'agosto': '8', 'septiembre': '9',
               'setiembre': '9', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'}
# Coherencia (solo control, no filtra): año y meses esperados por corte.
MESES_CORTE = {'2023_II': (2023, list(range(7, 13))),
               '2024_I': (2024, list(range(1, 7))),
               '2024_II': (2024, list(range(7, 13)))}

# cod_motesp: 2024_I lo publica sin ceros a la izquierda ('10304' frente a '010304').
# Se rellena a 6 dígitos para no partir un mismo motivo en dos grupos; el valor
# recibido queda en cod_motesp_origen.
ANCHO_COD_MOTESP = 6
VERSION_REGLAS = 'PD06-reglas-v1'
COLUMNAS_DERIVADAS = ['corte_archivo', 'cod_motesp_origen', 'codigo_departamento',
                      'departamento_valido', 'edad_anios', 'edad_valida', 'anio',
                      'mes_num', 'fecha_valida', 'fecha_en_corte']


# 2.2 Inspección: cinco filas por CSV, todas las columnas como texto

COLUMNAS_ORIGINALES = None
for periodo, csv in zip(PERIODOS, ARCHIVOS_CSV):
    muestra = pl.read_csv(csv, separator=SEPARADORES[periodo], n_rows=5, infer_schema=False)
    muestra = muestra.rename({c: c.lstrip('﻿') for c in muestra.columns})
    assert muestra.width == 38, f'{periodo}: {muestra.width} columnas'
    assert all(tipo == pl.String for tipo in muestra.dtypes), f'{periodo}: columnas no texto'
    if COLUMNAS_ORIGINALES is None:
        COLUMNAS_ORIGINALES = muestra.columns
    assert muestra.columns == COLUMNAS_ORIGINALES, f'{periodo}: contrato de columnas distinto'
    print(f'{csv.name} · separador {SEPARADORES[periodo]!r} · {muestra.width} columnas texto')
    display(muestra.select(['periodo', 'mes', 'afec_edad', 'afec_cod_depto',
                            'cod_motesp', 'patologia_1', 'pqr_canal']))
COLUMNAS_PARQUET = COLUMNAS_ORIGINALES + COLUMNAS_DERIVADAS


# 2.3 Plan lazy de normalización por corte (scan_csv + with_columns)

def plan_normalizado(periodo: str, csv: Path) -> pl.LazyFrame:
    lf = pl.scan_csv(csv, separator=SEPARADORES[periodo], infer_schema=False)
    con_bom = {c: c.lstrip('﻿') for c in lf.collect_schema().names() if c.startswith('﻿')}
    if con_bom:
        lf = lf.rename(con_bom)

    depto_txt = pl.col('afec_cod_depto').str.strip_chars()
    depto_2d = pl.when(depto_txt.str.contains(r'^\d{1,2}$')).then(depto_txt.str.zfill(2))
    depto_valido = depto_2d.is_in(list(DEPARTAMENTOS)).fill_null(False)

    edad_num = (pl.col('afec_edad').str.strip_chars()
                .str.replace(',', '.', literal=True).cast(pl.Float64, strict=False))
    edad_valida = (edad_num.is_finite() & (edad_num >= 0)
                   & (edad_num <= EDAD_MAXIMA)).fill_null(False)

    anio_num = pl.col('periodo').str.strip_chars().cast(pl.Int32, strict=False)
    mes_txt = pl.col('mes').str.strip_chars().str.to_lowercase().replace(MESES_TEXTO)
    mes_num = mes_txt.cast(pl.Int32, strict=False)
    fecha_valida = (anio_num.is_between(ANIO_MINIMO, ANIO_MAXIMO)
                    & mes_num.is_between(1, 12)).fill_null(False)
    anio_corte, meses_corte = MESES_CORTE[periodo]

    motivo_txt = pl.col('cod_motesp').str.strip_chars()
    return (lf.with_columns(
                pl.lit(periodo).alias('corte_archivo'),
                pl.col('cod_motesp').alias('cod_motesp_origen'),
                pl.when(motivo_txt.str.contains(rf'^\d{{1,{ANCHO_COD_MOTESP}}}$'))
                  .then(motivo_txt.str.zfill(ANCHO_COD_MOTESP))
                  .otherwise(motivo_txt).alias('cod_motesp'),
                pl.when(depto_valido).then(depto_2d).alias('codigo_departamento'),
                depto_valido.alias('departamento_valido'),
                pl.when(edad_valida).then(edad_num.floor()).cast(pl.Int16).alias('edad_anios'),
                edad_valida.alias('edad_valida'),
                pl.when(fecha_valida).then(anio_num).cast(pl.Int16).alias('anio'),
                pl.when(fecha_valida).then(mes_num).cast(pl.Int8).alias('mes_num'),
                fecha_valida.alias('fecha_valida'),
                ((anio_num == anio_corte) & mes_num.is_in(meses_corte))
                  .fill_null(False).alias('fecha_en_corte'))
            .select(COLUMNAS_PARQUET))


# 2.4 Verificación de Parquet existentes antes de reutilizarlos

def firma_csv(ruta: Path) -> dict:
    estado = ruta.stat()
    return {'archivo': ruta.name, 'bytes': estado.st_size, 'mtime_ns': estado.st_mtime_ns}

def revisar_parquet(periodo: str, csv: Path, parquet: Path, filas: int) -> tuple[bool, str]:
    manifiesto = parquet.with_suffix('.json')
    if not (parquet.is_file() and manifiesto.is_file()):
        return False, 'no existe Parquet o manifiesto'
    try:
        info = json.loads(manifiesto.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False, 'manifiesto ilegible'
    if info.get('csv') != firma_csv(csv):
        return False, 'el CSV de origen cambió (tamaño o fecha)'
    if info.get('version_reglas') != VERSION_REGLAS or info.get('corte') != periodo:
        return False, 'cambiaron las reglas de normalización'
    try:
        esquema = pl.scan_parquet(parquet).collect_schema()
        n = pl.scan_parquet(parquet).select(pl.len()).collect().item()
    except Exception as error:
        return False, f'Parquet ilegible: {error}'
    if esquema.names() != COLUMNAS_PARQUET:
        return False, 'contrato de columnas distinto'
    if any(esquema[c] != pl.String for c in COLUMNAS_ORIGINALES):
        return False, 'alguna columna original dejó de ser texto'
    if n != filas:
        return False, f'{n:,} filas; se esperaban {filas:,}'
    return True, 'Parquet verificado'

def escribir_parquet(plan: pl.LazyFrame, destino: Path) -> None:
    temporal = destino.parent / (destino.name + '.tmp')
    try:  # Sink en streaming CPU, Zstandard
        plan.sink_parquet(temporal, compression='zstd', engine='streaming')
    except TypeError:  # Polars sin parámetro engine en sinks: el sink ya es streaming
        plan.sink_parquet(temporal, compression='zstd')
    temporal.replace(destino)  # renombrado atómico: nunca queda un Parquet a medias


# 2.5 Conversión (CPU) con tiempo propio, separado del benchmark

conversion = []
for periodo, csv, parquet, filas in zip(PERIODOS, ARCHIVOS_CSV, ARCHIVOS_PARQUET, FILAS_ESPERADAS):
    reutilizable, motivo = revisar_parquet(periodo, csv, parquet, filas)
    segundos = 0.0
    if not reutilizable:
        print(f'{periodo}: se convierte ({motivo})')
        inicio = perf_counter()
        escribir_parquet(plan_normalizado(periodo, csv), parquet)
        segundos = perf_counter() - inicio
        parquet.with_suffix('.json').write_text(json.dumps({
            'corte': periodo, 'csv': firma_csv(csv), 'version_reglas': VERSION_REGLAS,
            'separador': SEPARADORES[periodo], 'polars': pl.__version__,
            'segundos_conversion': segundos}, indent=2), encoding='utf-8')
        reutilizable, motivo = revisar_parquet(periodo, csv, parquet, filas)
        assert reutilizable, f'{periodo}: conversión no verificada ({motivo})'
        accion = 'convertido'
    else:
        accion = 'reutilizado'
    conversion.append({'corte': periodo, 'accion': accion, 'control': motivo,
                       'segundos_conversion': round(segundos, 3), 'filas': filas,
                       'mb_csv': round(csv.stat().st_size / 2**20, 1),
                       'mb_parquet': round(parquet.stat().st_size / 2**20, 1)})
conversion = pl.DataFrame(conversion)
conversion.write_csv(OUT / 'conversion_PD06.csv')
display(conversion)


# 2.6 Calidad por corte y coherencia código ↔ descripción del motivo

calidad = (pl.scan_parquet(ARCHIVOS_PARQUET)
           .group_by('corte_archivo')
           .agg(pl.len().alias('reportes'),
                (pl.col('afec_cod_depto').str.strip_chars().str.len_chars() == 1).sum()
                  .alias('depto_sin_cero_origen'),
                (~pl.col('departamento_valido')).sum().alias('depto_invalido'),
                (~pl.col('edad_valida')).sum().alias('edad_nula'),
                (~pl.col('fecha_valida')).sum().alias('fecha_invalida'),
                (~pl.col('fecha_en_corte')).sum().alias('fecha_fuera_de_corte'),
                (pl.col('cod_motesp') != pl.col('cod_motesp_origen')).sum()
                  .alias('cod_motesp_rellenado'),
                pl.col('codigo_departamento').drop_nulls().n_unique().alias('departamentos'),
                pl.col('edad_anios').min().alias('edad_min'),
                pl.col('edad_anios').max().alias('edad_max'))
           .sort('corte_archivo').collect())
display(calidad)
calidad.write_csv(OUT / 'calidad_por_corte_PD06.csv')

mapa_motivos = (pl.scan_parquet(ARCHIVOS_PARQUET)
                .group_by('cod_motesp')
                .agg(pl.col('motivo_especifico').n_unique().alias('descripciones'),
                     pl.col('corte_archivo').n_unique().alias('cortes'))
                .collect())
assert (mapa_motivos['descripciones'] == 1).all(), 'Un código de motivo con varias descripciones'
print(f'{mapa_motivos.height} códigos de motivo específico; cada uno con una sola descripción.')

PQRD_2023_II.csv · separador ';' · 38 columnas texto


periodo,mes,afec_edad,afec_cod_depto,cod_motesp,patologia_1,pqr_canal
str,str,str,str,str,str,str
"""2023""","""7""","""69""","""11""","""010103""","""ENFERMEDADES CRÓNICAS NO TRASM…","""Personalizado"""
"""2023""","""7""","""35""","""11""","""040402""","""ENFERMEDADES HUÉRFANAS""","""Web"""
"""2023""","""7""","""77""","""70""","""010301""","""ENFERMEDADES CRÓNICAS NO TRASM…","""Personalizado"""
"""2023""","""7""","""71""","""76""","""010104""","""PROBLEMAS RELACIONADOS CON FAC…","""Escrito"""
"""2023""","""7""","""48""","""08""","""010304""","""PROBLEMAS RELACIONADOS CON FAC…","""Personalizado"""


PQRD_2024_I.csv · separador ',' · 38 columnas texto


periodo,mes,afec_edad,afec_cod_depto,cod_motesp,patologia_1,pqr_canal
str,str,str,str,str,str,str
"""2024""","""6""","""73""","""5""","""10304""","""ENFERMEDADES OSTEOARTICULARES""","""Telefónico"""
"""2024""","""4""","""84""","""63""","""10204""","""PROBLEMAS RELACIONADOS CON FAC…","""Redes Sociales"""
"""2024""","""5""","""0""","""11""","""10103""","""MATERNO INFANTIL""","""Web"""
"""2024""","""6""","""62""","""11""","""10301""","""ENFERMEDADES OSTEOARTICULARES""","""Web"""
"""2024""","""6""","""39""","""66""","""10103""","""CÁNCER ""","""Personalizado"""


PQRD_2024_II.csv · separador ';' · 38 columnas texto


periodo,mes,afec_edad,afec_cod_depto,cod_motesp,patologia_1,pqr_canal
str,str,str,str,str,str,str
"""2024""","""7""","""71""","""05""","""010101""","""ENFERMEDADES CRÓNICAS NO TRASM…","""Web"""
"""2024""","""7""","""61""","""11""","""010203""","""PROBLEMAS RELACIONADOS CON FAC…","""Web"""
"""2024""","""7""","""32""","""66""","""010101""","""ENFERMEDADES CRÓNICAS NO TRANS…","""Web"""
"""2024""","""7""","""9""","""66""","""010101""","""PROBLEMAS RELACIONADOS CON FAC…","""Web"""
"""2024""","""7""","""33""","""05""","""010103""","""PROBLEMAS RELACIONADOS CON FAC…","""Telefónico"""


corte,accion,control,segundos_conversion,filas,mb_csv,mb_parquet
str,str,str,f64,i64,f64,f64
"""2023_II""","""reutilizado""","""Parquet verificado""",0.0,716697,508.0,16.4
"""2024_I""","""reutilizado""","""Parquet verificado""",0.0,781601,550.0,18.1
"""2024_II""","""reutilizado""","""Parquet verificado""",0.0,946468,674.1,21.4


corte_archivo,reportes,depto_sin_cero_origen,depto_invalido,edad_nula,fecha_invalida,fecha_fuera_de_corte,cod_motesp_rellenado,departamentos,edad_min,edad_max
str,u32,u32,u32,u32,u32,u32,u32,u32,i16,i16
"""2023_II""",716697,0,0,0,0,0,0,33,0,113
"""2024_I""",781601,167477,0,1,0,0,781601,33,0,113
"""2024_II""",946468,0,0,0,0,0,0,33,0,118


37 códigos de motivo específico; cada uno con una sola descripción.


In [6]:
# Control de preparación; ejecutar después de completar la celda anterior.
assert all(p.is_file() for p in ARCHIVOS_PARQUET), 'Faltan periodos convertidos'
filas_por_archivo = [pl.scan_parquet(p).select(pl.len()).collect().item()
                     for p in ARCHIVOS_PARQUET]
assert filas_por_archivo == FILAS_ESPERADAS
assert sum(filas_por_archivo) == TOTAL_ESPERADO
REQUERIDAS = ['codigo_departamento', 'edad_anios', 'patologia_1',
              'cod_motesp', 'motivo_especifico', 'pqr_canal', 'anio', 'mes_num']
for archivo in ARCHIVOS_PARQUET:
    esquema = pl.scan_parquet(archivo).collect_schema()
    assert set(REQUERIDAS) <= set(esquema.names())
    assert esquema['anio'].is_integer() and esquema['mes_num'].is_integer()
    assert esquema['edad_anios'].is_numeric()
print('Tres periodos conservados:', filas_por_archivo)

Tres periodos conservados: [716697, 781601, 946468]


## 3. Ejercicio 1 · Departamentos con más quejas por perfil

**Combina todos los periodos completos**, no solo una muestra. Define el perfil:

| Dimensión | Campo |
|---|---|
| Edad en años | `edad_anios`, construida desde `afec_edad` |
| Patología | `patologia_1` |
| Código del motivo específico | `cod_motesp` |
| Descripción del mismo motivo | `motivo_especifico` |
| Canal de entrada | `pqr_canal` |

No mezcles un código de motivo específico con la descripción de otro nivel.
Agrupa por departamento y las cinco dimensiones; cuenta filas como `quejas`.
Ordena por perfil, quejas descendentes y código de departamento ascendente
para resolver empates de forma reproducible. Conserva los grupos no informados.

**Producto:** una tabla detallada con todas las combinaciones y sus conteos;
y una vista de hasta `TOP_N` departamentos con más quejas **para cada perfil**.
No basta un único top nacional que ignore edad, patología, motivo y canal.
Puedes preparar ese recorte después de medir, indicándolo en el informe.

El plan medido debe incluir lectura Parquet, agrupación y ordenamiento.
Declara un `LazyFrame` llamado `consulta_e1`; no lo ejecutes todavía.
No presentes quejas como personas únicas ni como tasas por población.


In [7]:
# Perfil = edad + patología + código y descripción del MISMO nivel (motivo específico) + canal.
DIM_PERFIL = ['edad_anios', 'patologia_1', 'cod_motesp', 'motivo_especifico', 'pqr_canal']
CLAVES_E1 = ['codigo_departamento', *DIM_PERFIL]

# Fuente común: los tres Parquet completos (2.444.766 reportes), sin filtros.
fuente = pl.scan_parquet(ARCHIVOS_PARQUET)

# Plan medido: lectura Parquet → proyección → group_by → conteo → orden total.
# Orden: perfil ascendente, quejas descendentes y departamento ascendente (desempate).
# nulls_last=True: los grupos no informados (edad o departamento nulos) se conservan al final.
consulta_e1 = (
    fuente
    .select(CLAVES_E1)
    .group_by(CLAVES_E1)
    .agg(pl.len().alias('quejas'))
    .sort([*DIM_PERFIL, 'quejas', 'codigo_departamento'],
          descending=[False] * len(DIM_PERFIL) + [True, False],
          nulls_last=True)
)
print(consulta_e1.explain())

SORT BY [descending: [false, false, false, false, false, true, false], nulls_last: [true, true, true, true, true, true, true]] [col("edad_anios"), col("patologia_1"), col("cod_motesp"), col("motivo_especifico"), col("pqr_canal"), col("quejas"), col("codigo_departamento")]
  AGGREGATE[maintain_order: false]
    [len().alias("quejas")] BY [col("codigo_departamento"), col("edad_anios"), col("patologia_1"), col("cod_motesp"), col("motivo_especifico"), col("pqr_canal")]
    FROM
    Parquet SCAN [/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06/parquet/2023_II.parquet, ... 2 other sources]
    PROJECT 6/48 COLUMNS
    ESTIMATED ROWS: 2150091


## 4. Ejercicio 2 · Evolución mensual de los mismos perfiles

Conserva departamento, edad, patología, código y descripción del motivo y canal.
Añade **año y mes numéricos** a la agrupación, cuenta las quejas y ordena el
resultado **primero por año y mes ascendentes**, después por las dimensiones
para que la salida sea reproducible. Enero, febrero y octubre deben quedar
como meses 1, 2 y 10, no como cadenas ordenadas alfabéticamente.

**Producto:** tabla `anio`, `mes_num`, departamento, las cinco dimensiones y
`quejas`, que permita seguir un mismo perfil mes a mes. Muestra dos perfiles
concretos y explica sus cambios. No concluyas que toda variación es tendencia.

Conserva las fechas inválidas en el control total y sepáralas al presentar la
cronología. No inventes meses ni rellenes silenciosamente ausencias con cero.
Declara el plan lazy como `consulta_e2`, sin ejecutarlo todavía.


In [8]:
# Mismas dimensiones del ejercicio 1 + año y mes numéricos (Int16 / Int8).
CLAVES_E2 = ['anio', 'mes_num', *CLAVES_E1]

# Orden cronológico ascendente (1, 2, …, 10, no '1', '10', '2'), después
# departamento y perfil para una salida reproducible. Fechas inválidas (nulas) al final:
# se cuentan en el total y se separan al presentar la cronología.
consulta_e2 = (
    fuente
    .select(CLAVES_E2)
    .group_by(CLAVES_E2)
    .agg(pl.len().alias('quejas'))
    .sort(CLAVES_E2, nulls_last=True)
)
print(consulta_e2.explain())

SORT BY [nulls_last: [true, true, true, true, true, true, true, true]] [col("anio"), col("mes_num"), col("codigo_departamento"), col("edad_anios"), col("patologia_1"), col("cod_motesp"), col("motivo_especifico"), col("pqr_canal")]
  AGGREGATE[maintain_order: false]
    [len().alias("quejas")] BY [col("anio"), col("mes_num"), col("codigo_departamento"), col("edad_anios"), col("patologia_1"), col("cod_motesp"), col("motivo_especifico"), col("pqr_canal")]
    FROM
    Parquet SCAN [/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06/parquet/2023_II.parquet, ... 2 other sources]
    PROJECT 8/48 COLUMNS
    ESTIMATED ROWS: 2150091


## 5. Contrato de comparación y frontera temporal

Completa las listas de claves y la comprobación para cada resultado.
Verifica todas las filas agregadas, no solo la suma global. Los conteos deben
ser exactos y cada consulta debe conservar 2.444.766 reportes, incluidos grupos
no informados. Revisa tipos, nulos y duplicados de claves agregadas.

Mediremos con `perf_counter`: **leer Parquet + agregar + ordenar + devolver un
DataFrame Polars en CPU**. `collect` termina esa frontera para ambos motores.
No se mide el tiempo de construir el plan, descargar, convertir, verificar ni
exportar. La medición GPU incluye transferencias; no es tiempo exclusivo de
kernels CUDA ni una comparación de datos ya residentes en VRAM.

CPU usará `engine='streaming'`. GPU usará `GPUEngine(raise_on_fail=True)`.
Si el plan no está soportado en GPU, corrígelo o informa el error; no lo midas
como GPU mediante fallback a CPU. No cambies filtros o claves entre motores.


In [9]:
assert isinstance(consulta_e1, pl.LazyFrame), 'Completar ejercicio 1'
assert isinstance(consulta_e2, pl.LazyFrame), 'Completar ejercicio 2'
CONSULTAS = {'e1_departamentos': consulta_e1, 'e2_evolucion_mensual': consulta_e2}
CLAVES_RESULTADO = {
    'e1_departamentos': CLAVES_E1,
    'e2_evolucion_mensual': CLAVES_E2,
}
assert all(CLAVES_RESULTADO.values()), 'Definir las claves para comparar todos los grupos'
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)

# Contrato estático (sin ejecutar el plan): columnas y tipos de cada resultado.
ESQUEMA_PARQUET = pl.scan_parquet(ARCHIVOS_PARQUET[0]).collect_schema()
for nombre, consulta in CONSULTAS.items():
    esquema = consulta.collect_schema()
    assert esquema.names() == CLAVES_RESULTADO[nombre] + ['quejas'], nombre
    assert all(esquema[c] == ESQUEMA_PARQUET[c] for c in CLAVES_RESULTADO[nombre]), nombre
    assert esquema['quejas'].is_integer(), nombre
    print(f'{nombre}: {dict(esquema)}')

def controlar_resultado(df: pl.DataFrame, nombre: str) -> dict:
    """Controles sobre TODAS las filas agregadas; se ejecuta siempre fuera del reloj."""
    claves = CLAVES_RESULTADO[nombre]
    assert df.columns == claves + ['quejas'], f'{nombre}: columnas inesperadas'
    assert all(df.schema[c] == ESQUEMA_PARQUET[c] for c in claves), f'{nombre}: tipos de clave'
    assert df['quejas'].null_count() == 0 and df['quejas'].min() >= 1, f'{nombre}: conteos'
    assert int(df['quejas'].sum()) == TOTAL_ESPERADO, f'{nombre}: no conserva el total'
    assert not df.select(claves).is_duplicated().any(), f'{nombre}: claves duplicadas'
    nulos = {c: {'grupos': int(df[c].null_count()),
                 'reportes': int(df.filter(pl.col(c).is_null())['quejas'].sum())}
             for c in claves}
    return {'grupos': df.height, 'reportes': int(df['quejas'].sum()),
            'claves': claves, 'claves_unicas': True, 'nulos_por_clave': nulos}

controles = {}
primeras = []

e1_departamentos: {'codigo_departamento': String, 'edad_anios': Int16, 'patologia_1': String, 'cod_motesp': String, 'motivo_especifico': String, 'pqr_canal': String, 'quejas': UInt32}
e2_evolucion_mensual: {'anio': Int16, 'mes_num': Int8, 'codigo_departamento': String, 'edad_anios': Int16, 'patologia_1': String, 'cod_motesp': String, 'motivo_especifico': String, 'pqr_canal': String, 'quejas': UInt32}


### Ejercicio 1 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [10]:
from time import perf_counter

# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e1 = consulta_e1.collect(engine='streaming')
tiempo_cpu_e1 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e1 = consulta_e1.collect(engine=engine_gpu)
tiempo_gpu_e1 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']),
                   gpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']), check_exact=True)
assert cpu_e1['quejas'].sum() == TOTAL_ESPERADO
controles['e1_departamentos'] = controlar_resultado(cpu_e1, 'e1_departamentos')
controles['e1_departamentos']['orden_identico_cpu_gpu'] = cpu_e1.equals(gpu_e1)
primeras.extend([
    {'ejercicio': 'e1', 'motor': 'cpu', 'segundos': tiempo_cpu_e1},
    {'ejercicio': 'e1', 'motor': 'gpu', 'segundos': tiempo_gpu_e1}])
print('Ejercicio 1 · CPU:', tiempo_cpu_e1, 's · GPU:', tiempo_gpu_e1, 's')
print('Grupos:', cpu_e1.height, '· orden idéntico CPU/GPU:',
      controles['e1_departamentos']['orden_identico_cpu_gpu'])
display(gpu_e1.head(20))

Ejercicio 1 · CPU: 0.6941981890704483 s · GPU: 3.167635352117941 s
Grupos: 571235 · orden idéntico CPU/GPU: True


codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas
str,i16,str,str,str,str,u32
"""76""",0,"""COVID -19""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Chat""",1
"""11""",0,"""COVID -19""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Web""",1
"""08""",0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Personalizado""",1
"""05""",0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico""",1
"""11""",0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Web""",1
…,…,…,…,…,…,…
"""11""",0,"""COVID -19""","""020101""","""INSATISFACCIÓN POR FALTA DE OP…","""Web""",1
"""11""",0,"""COVID -19""","""020201""","""INSATISFACCIÓN RELACIONADA CON…","""Escrito""",1
"""05""",0,"""CUIDADO INTENSIVO PARA CUALQUI…","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Personalizado""",1


### Ejercicio 2 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [14]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e2 = consulta_e2.collect(engine='streaming')
tiempo_cpu_e2 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e2 = consulta_e2.collect(engine=engine_gpu)
tiempo_gpu_e2 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']),
                   gpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']), check_exact=True)
assert cpu_e2['quejas'].sum() == TOTAL_ESPERADO
controles['e2_evolucion_mensual'] = controlar_resultado(cpu_e2, 'e2_evolucion_mensual')
controles['e2_evolucion_mensual']['orden_identico_cpu_gpu'] = cpu_e2.equals(gpu_e2)
# Orden cronológico verificado sobre las fechas válidas: (anio, mes_num) no decrece.
periodo_ordinal = cpu_e2.drop_nulls(['anio', 'mes_num']).select(
    pl.col('anio').cast(pl.Int32) * 12 + pl.col('mes_num').cast(pl.Int32))
assert (periodo_ordinal.to_series().diff().drop_nulls() >= 0).all(), 'Orden cronológico roto'
primeras.extend([
    {'ejercicio': 'e2', 'motor': 'cpu', 'segundos': tiempo_cpu_e2},
    {'ejercicio': 'e2', 'motor': 'gpu', 'segundos': tiempo_gpu_e2}])
print('Ejercicio 2 · CPU:', tiempo_cpu_e2, 's · GPU:', tiempo_gpu_e2, 's')
print('Grupos:', cpu_e2.height, '· orden idéntico CPU/GPU:',
      controles['e2_evolucion_mensual']['orden_identico_cpu_gpu'])
display(gpu_e2.head(20))

Ejercicio 2 · CPU: 1.6316508599556983 s · GPU: 2.9914760331157595 s
Grupos: 1492458 · orden idéntico CPU/GPU: True


anio,mes_num,codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas
i16,i8,str,i16,str,str,str,str,u32
2023,7,"""05""",0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico""",1
2023,7,"""05""",0,"""ENFERMEDADES CRÓNICAS NO TRASM…","""010302""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Telefónico""",1
2023,7,"""05""",0,"""ENFERMEDADES CRÓNICAS NO TRASM…","""010304""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Web""",1
2023,7,"""05""",0,"""ENFERMEDADES HUÉRFANAS""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Personalizado""",1
2023,7,"""05""",0,"""ENFERMEDADES HUÉRFANAS""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Chat""",1
…,…,…,…,…,…,…,…,…
2023,7,"""05""",0,"""MATERNO INFANTIL""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico""",8
2023,7,"""05""",0,"""MATERNO INFANTIL""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Web""",11
2023,7,"""05""",0,"""MATERNO INFANTIL""","""010104""","""FALTA DE OPORTUNIDAD EN LA ENT…","""Telefónico""",3


## 6. Calentamiento y tres repeticiones por motor

Completa el bucle de medición. Por cada ejercicio:

1. Ejecuta una vuelta adicional de calentamiento CPU/GPU y no la mezcles con las medidas.
2. Repite al menos tres veces, alternando CPU→GPU y GPU→CPU.
3. Rodea cada `collect` con `perf_counter`; compara resultados después de parar el reloj.
4. Guarda `ejercicio`, `motor`, `repeticion`, `segundos`, `filas_entrada`,
   `grupos_resultado` y `equivalente` en la lista `mediciones`.
5. Revisa que los resultados conserven todos los reportes y las claves esperadas.

Mide los mismos planes usados arriba. Mantén las mismas condiciones y registra
hilos CPU, GPU, VRAM, controlador y versiones. No se borran automáticamente las
cachés del sistema operativo; explica su efecto. No uses `process_time` como
sustituto de la duración total de la consulta GPU.


### Registro del entorno

Se documentan versiones, CPU, hilos de Polars y GPU (modelo, VRAM, controlador)
antes de medir, con las mismas condiciones para ambos motores.

In [11]:
# Registro del entorno de medición (fuera del reloj).
def version_paquete(*nombres: str) -> str | None:
    for nombre in nombres:
        try:
            return f'{nombre}=={metadata.version(nombre)}'
        except metadata.PackageNotFoundError:
            continue
    return None

def modelo_cpu() -> str:
    try:
        for linea in Path('/proc/cpuinfo').read_text().splitlines():
            if linea.startswith('model name'):
                return linea.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

def info_gpu(indice: int) -> dict:
    consulta = 'index,name,memory.total,memory.used,driver_version'
    try:
        salida = subprocess.run(
            ['nvidia-smi', f'--query-gpu={consulta}', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True, timeout=30).stdout
        cabecera = subprocess.run(['nvidia-smi'], capture_output=True, text=True,
                                  timeout=30).stdout
    except (OSError, subprocess.SubprocessError) as error:
        return {'error': f'nvidia-smi no disponible: {error}'}
    for fila in salida.strip().splitlines():
        idx, nombre, total, usada, driver = [x.strip() for x in fila.split(',')]
        if int(idx) == indice:
            cuda = next((p.split(':')[1].strip().split()[0] for p in cabecera.split('|')
                         if 'CUDA Version' in p), None)
            return {'indice': indice, 'modelo': nombre, 'vram_total_mib': int(total),
                    'vram_usada_mib_al_registrar': int(usada), 'controlador': driver,
                    'cuda_controlador': cuda}
    return {'error': f'No existe la GPU {indice}'}

entorno = {
    'python': sys.version.split()[0],
    'plataforma': platform.platform(),
    'polars': pl.__version__,
    'cudf_polars': version_paquete('cudf-polars-cu12', 'cudf-polars-cu13', 'cudf-polars'),
    'rmm': version_paquete('rmm-cu12', 'rmm-cu13', 'rmm'),
    'cpu_modelo': modelo_cpu(),
    'cpu_nucleos_logicos': os.cpu_count(),
    'polars_hilos': pl.thread_pool_size(),
    'gpu': info_gpu(DISPOSITIVO_CUDA),
    'motor_cpu': "collect(engine='streaming')",
    'motor_gpu': f'GPUEngine(device={DISPOSITIVO_CUDA}, raise_on_fail=True)',
}
print(json.dumps(entorno, indent=2, ensure_ascii=False))

{
  "python": "3.12.15",
  "plataforma": "Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.43",
  "polars": "1.35.2",
  "cudf_polars": "cudf-polars-cu12==26.10.0",
  "rmm": "rmm-cu12==26.10.0",
  "cpu_modelo": "13th Gen Intel(R) Core(TM) i7-13700HX",
  "cpu_nucleos_logicos": 24,
  "polars_hilos": 24,
  "gpu": {
    "indice": 0,
    "modelo": "NVIDIA RTX A1000 6GB Laptop GPU",
    "vram_total_mib": 6144,
    "vram_usada_mib_al_registrar": 741,
    "controlador": "596.58",
    "cuda_controlador": "596.58"
  },
  "motor_cpu": "collect(engine='streaming')",
  "motor_gpu": "GPUEngine(device=0, raise_on_fail=True)"
}


In [15]:
mediciones = []
calentamiento = []
MOTORES = {'cpu': 'streaming', 'gpu': engine_gpu}
REFERENCIAS = {'e1_departamentos': cpu_e1, 'e2_evolucion_mensual': cpu_e2}
del gpu_e1, gpu_e2  # libera memoria; las referencias CPU ya fueron validadas contra GPU

def es_equivalente(df: pl.DataFrame, nombre: str) -> bool:
    """Compara TODAS las filas contra la referencia validada; nunca dentro del reloj."""
    claves = CLAVES_RESULTADO[nombre]
    try:
        assert_frame_equal(REFERENCIAS[nombre].sort(claves), df.sort(claves), check_exact=True)
    except AssertionError:
        return False
    return int(df['quejas'].sum()) == TOTAL_ESPERADO and df.columns == claves + ['quejas']

for nombre, consulta in CONSULTAS.items():
    # 1) Calentamiento CPU y GPU: se registra aparte y no entra en las medianas.
    for motor in ('cpu', 'gpu'):
        inicio = perf_counter()
        resultado = consulta.collect(engine=MOTORES[motor])
        segundos = perf_counter() - inicio
        calentamiento.append({'ejercicio': nombre, 'motor': motor, 'segundos': segundos,
                              'equivalente': es_equivalente(resultado, nombre)})
        del resultado

    # 2) Repeticiones alternando el orden: impar CPU→GPU, par GPU→CPU.
    for repeticion in range(1, REPETICIONES + 1):
        orden = ('cpu', 'gpu') if repeticion % 2 == 1 else ('gpu', 'cpu')
        for motor in orden:
            inicio = perf_counter()                               # ── reloj ──
            resultado = consulta.collect(engine=MOTORES[motor])   # leer+agregar+ordenar
            segundos = perf_counter() - inicio                    # ── fin reloj ──
            equivalente = es_equivalente(resultado, nombre)       # fuera del reloj
            mediciones.append({
                'ejercicio': nombre, 'motor': motor, 'repeticion': repeticion,
                'orden': '→'.join(orden), 'segundos': segundos,
                'filas_entrada': int(resultado['quejas'].sum()),
                'grupos_resultado': resultado.height, 'equivalente': equivalente})
            assert equivalente, f'{nombre} · {motor} · rep {repeticion}: resultado distinto'
            del resultado
        print(f'{nombre} · repetición {repeticion} ({"→".join(orden)}):',
              ' · '.join(f"{m['motor']} {m['segundos']:.3f} s" for m in mediciones[-2:]))

pl.DataFrame(calentamiento).write_csv(OUT / 'calentamiento_PD06.csv')
display(pl.DataFrame(calentamiento))
display(pl.DataFrame(mediciones))

e1_departamentos · repetición 1 (cpu→gpu): cpu 0.549 s · gpu 0.734 s
e1_departamentos · repetición 2 (gpu→cpu): gpu 0.680 s · cpu 0.551 s
e1_departamentos · repetición 3 (cpu→gpu): cpu 0.570 s · gpu 0.901 s
e2_evolucion_mensual · repetición 1 (cpu→gpu): cpu 0.884 s · gpu 1.456 s
e2_evolucion_mensual · repetición 2 (gpu→cpu): gpu 1.068 s · cpu 0.936 s
e2_evolucion_mensual · repetición 3 (cpu→gpu): cpu 1.479 s · gpu 1.570 s


ejercicio,motor,segundos,equivalente
str,str,f64,bool
"""e1_departamentos""","""cpu""",0.607137,true
"""e1_departamentos""","""gpu""",1.969303,true
"""e2_evolucion_mensual""","""cpu""",0.964902,true
"""e2_evolucion_mensual""","""gpu""",1.553248,true


ejercicio,motor,repeticion,orden,segundos,filas_entrada,grupos_resultado,equivalente
str,str,i64,str,f64,i64,i64,bool
"""e1_departamentos""","""cpu""",1,"""cpu→gpu""",0.549113,2444766,571235,true
"""e1_departamentos""","""gpu""",1,"""cpu→gpu""",0.73359,2444766,571235,true
"""e1_departamentos""","""gpu""",2,"""gpu→cpu""",0.679797,2444766,571235,true
"""e1_departamentos""","""cpu""",2,"""gpu→cpu""",0.551185,2444766,571235,true
"""e1_departamentos""","""cpu""",3,"""cpu→gpu""",0.569856,2444766,571235,true
…,…,…,…,…,…,…,…
"""e2_evolucion_mensual""","""gpu""",1,"""cpu→gpu""",1.455966,2444766,1492458,true
"""e2_evolucion_mensual""","""gpu""",2,"""gpu→cpu""",1.067738,2444766,1492458,true
"""e2_evolucion_mensual""","""cpu""",2,"""gpu→cpu""",0.935833,2444766,1492458,true


In [1]:
import polars as pl

df = pl.DataFrame({"x": [1, 2, 3]})

resultado = df.lazy().select(
    pl.col("x").sum()
).collect(engine="gpu")

display(resultado)

x
i64
6


## 7. Tabla de tiempos y factor de aceleración

La celda siguiente no resuelve consultas: resume tus mediciones. Habrá
**12 filas como mínimo**: dos ejercicios × dos motores × tres repeticiones.
Factor CPU/GPU = mediana CPU / mediana GPU: mayor que 1 favorece GPU;
menor que 1 favorece CPU. No mezcles ambas consultas ni las primeras ejecuciones.


In [16]:
assert len(mediciones) == len(CONSULTAS) * 2 * REPETICIONES
requeridas = {'ejercicio', 'motor', 'repeticion', 'segundos',
              'filas_entrada', 'grupos_resultado', 'equivalente'}
assert all(requeridas <= set(fila) for fila in mediciones)
tiempos = pl.DataFrame(mediciones)
assert tiempos['equivalente'].all()
assert (tiempos['filas_entrada'] == TOTAL_ESPERADO).all()
assert tiempos['segundos'].is_finite().all() and (tiempos['segundos'] > 0).all()
assert set(tiempos['ejercicio'].to_list()) == set(CONSULTAS)
assert set(tiempos['motor'].to_list()) == {'cpu', 'gpu'}
conteos = tiempos.group_by(['ejercicio', 'motor']).agg(pl.len().alias('n'))
assert (conteos['n'] == REPETICIONES).all()
assert tiempos.select(['ejercicio', 'motor', 'repeticion']).unique().height == tiempos.height
resumen = tiempos.group_by(['ejercicio', 'motor']).agg(
    pl.col('segundos').median().alias('mediana_s'),
    pl.col('segundos').min().alias('min_s'),
    pl.col('segundos').max().alias('max_s')).sort(['ejercicio', 'motor'])
factores = []
for nombre in CONSULTAS:
    cpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'cpu'))['mediana_s'].item()
    gpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'gpu'))['mediana_s'].item()
    factores.append({'ejercicio': nombre, 'mediana_cpu_s': cpu_s,
                     'mediana_gpu_s': gpu_s, 'factor_cpu_sobre_gpu': cpu_s / gpu_s})
display(resumen)
display(pl.DataFrame(factores))
tiempos.write_csv(OUT / 'benchmark_PD06.csv')
resumen.write_csv(OUT / 'resumen_PD06.csv')
pl.DataFrame(primeras).write_csv(OUT / 'primeras_ejecuciones_PD06.csv')
pl.DataFrame(factores).write_csv(OUT / 'factores_PD06.csv')

ejercicio,motor,mediana_s,min_s,max_s
str,str,f64,f64,f64
"""e1_departamentos""","""cpu""",0.551185,0.549113,0.569856
"""e1_departamentos""","""gpu""",0.73359,0.679797,0.900559
"""e2_evolucion_mensual""","""cpu""",0.935833,0.883864,1.479128
"""e2_evolucion_mensual""","""gpu""",1.455966,1.067738,1.569602


ejercicio,mediana_cpu_s,mediana_gpu_s,factor_cpu_sobre_gpu
str,f64,f64,f64
"""e1_departamentos""",0.551185,0.73359,0.751354
"""e2_evolucion_mensual""",0.935833,1.455966,0.642757


## 8. Interpretar y entregar

Completa el notebook y entrega en tu repositorio:

- `soluciones/PD06_PQRS_agregaciones_CPU_GPU.ipynb` con código visible y conclusiones.
- Resultados regenerables en `kit/salidas/soluciones/PD06/`: detalle/ranking de
  departamentos, evolución mensual, controles, tiempos, resumen y factores.
- Un control JSON con filas por corte, total conciliado, número de grupos por
  consulta, reglas de edad y fechas, cobertura de departamentos, versiones,
  modelo GPU/VRAM/controlador, hilos CPU y alcance de cada medición.

**Responde:** ¿qué departamentos lideran dentro de cada perfil? ¿Cómo evolucionan
los dos perfiles seleccionados? ¿Cuántos grupos genera cada consulta? ¿La GPU
acelera ambas? ¿Qué parte puede estar limitada por lectura, transferencias,
ordenamiento o memoria? ¿Cambió la conclusión entre primera ejecución y mediana?

Se evalúan corrección, equivalencia, medición reproducible e interpretación;
no que GPU necesariamente gane. No adjuntes los CSV originales al repositorio.
La salida queda en la ruta ignorada `kit/salidas/`. La copia resuelta pertenece
al estudiante; esta plantilla mantiene sus celdas por completar.

Apoyo: [guía Polars](../docs/manual-polars.md),
[entorno GPU](../docs/polars-gpu.md),
[referencia P04](../talleres/P04.md). Son referencias, no una solución de PD06.


## 9. Resultados para el informe · fuera del reloj

Las vistas siguientes se preparan **después de medir**, a partir del resultado CPU
ya validado contra GPU (son idénticos). No forman parte del tiempo medido.

### 9.1 Ejercicio 1 · Top de departamentos dentro de cada perfil

In [17]:
# ── Ejercicio 1 · vistas para el informe (fuera del reloj, sobre el resultado validado) ──
def nombre_departamento() -> pl.Expr:
    return (pl.col('codigo_departamento')
            .replace_strict(DEPARTAMENTOS, default=None, return_dtype=pl.String)
            .fill_null('No informado o inválido').alias('departamento'))

CLAVE_PERFIL = pl.concat_str([pl.col(c).cast(pl.String).fill_null('∅') for c in DIM_PERFIL],
                             separator=' | ').alias('clave_perfil')

detalle_e1 = cpu_e1.with_columns(nombre_departamento())

# cpu_e1 ya viene ordenado por perfil, quejas desc y departamento asc (orden del plan medido):
# la posición dentro de cada perfil es el número de fila dentro de su ventana.
ranking_e1 = (cpu_e1
    .with_columns(pl.int_range(1, pl.len() + 1).over(DIM_PERFIL).alias('posicion'),
                  pl.col('quejas').sum().over(DIM_PERFIL).alias('quejas_perfil'),
                  pl.len().over(DIM_PERFIL).alias('departamentos_con_quejas'),
                  # Empate con el siguiente: el desempate por código favorece códigos bajos (p. ej. 05).
                  (pl.col('quejas') == pl.col('quejas').shift(-1).over(DIM_PERFIL))
                    .fill_null(False).alias('empata_con_siguiente'))
    .filter(pl.col('posicion') <= TOP_N)
    .with_columns(nombre_departamento(),
                  (pl.col('quejas') / pl.col('quejas_perfil')).round(4)
                    .alias('participacion_en_perfil')))

# Control del ranking: la posición 1 de cada perfil es su máximo y no hay más de TOP_N por perfil.
chequeo = ranking_e1.group_by(DIM_PERFIL).agg(
    pl.col('quejas').max().alias('maximo'),
    pl.col('quejas').filter(pl.col('posicion') == 1).first().alias('primero'),
    pl.len().alias('filas'))
assert (chequeo['maximo'] == chequeo['primero']).all() and chequeo['filas'].max() <= TOP_N
n_perfiles = chequeo.height

# Perfiles con más quejas y sus tres departamentos líderes.
PERFILES_A_MOSTRAR = 15
top_perfiles = (ranking_e1.filter(pl.col('posicion') == 1)
                .sort(['quejas_perfil', *DIM_PERFIL],
                      descending=[True] + [False] * len(DIM_PERFIL), nulls_last=True)
                .head(PERFILES_A_MOSTRAR).with_columns(CLAVE_PERFIL)['clave_perfil'].to_list())
lideres_e1 = (ranking_e1.with_columns(CLAVE_PERFIL)
    .filter(pl.col('clave_perfil').is_in(top_perfiles) & (pl.col('posicion') <= 3))
    .group_by('clave_perfil', maintain_order=True)
    .agg(*[pl.col(c).first() for c in DIM_PERFIL],
         pl.col('quejas_perfil').first(), pl.col('departamentos_con_quejas').first(),
         pl.format('{} ({} · {}%)', 'departamento', 'quejas',
                   (pl.col('participacion_en_perfil') * 100).round(1)).alias('top_3'))
    .with_columns(pl.col('top_3').list.join(' · '))
    .sort('quejas_perfil', descending=True)
    .drop('clave_perfil'))

# Cuántos perfiles encabeza cada departamento (posición 1) y con cuántas quejas.
liderazgo = (ranking_e1.filter(pl.col('posicion') == 1)
    .group_by('codigo_departamento', 'departamento')
    .agg(pl.len().alias('perfiles_que_lidera'),
         (~pl.col('empata_con_siguiente')).sum().alias('lidera_sin_empate'),
         pl.col('quejas').sum().alias('quejas_como_lider'))
    .with_columns((pl.col('perfiles_que_lidera') / n_perfiles * 100).round(2).alias('pct_perfiles'))
    .sort(['perfiles_que_lidera', 'codigo_departamento'], descending=[True, False], nulls_last=True))

concentracion = (ranking_e1.filter(pl.col('posicion') == 1).select(
    pl.len().alias('perfiles'),
    (pl.col('quejas_perfil') == 1).sum().alias('perfiles_con_1_queja'),
    (pl.col('quejas_perfil') >= 100).sum().alias('perfiles_con_100_o_mas'),
    pl.col('quejas_perfil').median().alias('mediana_quejas_por_perfil'),
    (pl.col('departamentos_con_quejas') == 1).sum().alias('perfiles_en_un_solo_departamento'),
    pl.col('empata_con_siguiente').sum().alias('perfiles_con_empate_en_primer_lugar')))

print(f'Combinaciones departamento × perfil: {cpu_e1.height:,} · perfiles distintos: {n_perfiles:,}')
display(concentracion)
with pl.Config(fmt_str_lengths=120, tbl_width_chars=260, tbl_rows=PERFILES_A_MOSTRAR):
    display(lideres_e1)
with pl.Config(tbl_rows=12):
    display(liderazgo.head(12))

detalle_e1.write_parquet(OUT / 'e1_detalle_departamento_perfil.parquet')
ranking_e1.write_parquet(OUT / f'e1_ranking_top{TOP_N}_por_perfil.parquet')
lideres_e1.write_csv(OUT / 'e1_perfiles_principales_top3.csv')
liderazgo.write_csv(OUT / 'e1_liderazgo_departamentos.csv')

Combinaciones departamento × perfil: 571,235 · perfiles distintos: 113,803


perfiles,perfiles_con_1_queja,perfiles_con_100_o_mas,mediana_quejas_por_perfil,perfiles_en_un_solo_departamento,perfiles_con_empate_en_primer_lugar
u32,u32,u32,f64,u32,u32
113803,38020,4621,3.0,41101,31616


edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas_perfil,departamentos_con_quejas,top_3
i16,str,str,str,str,u32,u32,str
0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",3091,31,"""Bogotá D. C. (886 · 28.7%) · Antioquia (357 · 11.6%) · Cundinamarca (314 · 10.2%)"""
0,"""MATERNO INFANTIL""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CITAS O CONSULTAS""","""Web""",2771,31,"""Bogotá D. C. (711 · 25.7%) · Antioquia (416 · 15.0%) · Valle del Cauca (248 · 9.0%)"""
63,"""PROBLEMAS RELACIONADOS CON FACILIDADES DE ATENCIÓN MÉDICA U OTROS SERVICIOS DE SALUD""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2592,27,"""Bogotá D. C. (803 · 31.0%) · Antioquia (519 · 20.0%) · Valle del Cauca (223 · 8.6%)"""
4,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2588,32,"""Bogotá D. C. (726 · 28.1%) · Antioquia (533 · 20.6%) · Valle del Cauca (211 · 8.2%)"""
61,"""PROBLEMAS RELACIONADOS CON FACILIDADES DE ATENCIÓN MÉDICA U OTROS SERVICIOS DE SALUD""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2538,28,"""Bogotá D. C. (700 · 27.6%) · Antioquia (536 · 21.1%) · Valle del Cauca (222 · 8.8%)"""
60,"""PROBLEMAS RELACIONADOS CON FACILIDADES DE ATENCIÓN MÉDICA U OTROS SERVICIOS DE SALUD""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2515,30,"""Bogotá D. C. (656 · 26.1%) · Antioquia (515 · 20.5%) · Valle del Cauca (245 · 9.7%)"""
1,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2483,30,"""Bogotá D. C. (737 · 29.7%) · Antioquia (373 · 15.0%) · Cundinamarca (216 · 8.7%)"""
58,"""PROBLEMAS RELACIONADOS CON FACILIDADES DE ATENCIÓN MÉDICA U OTROS SERVICIOS DE SALUD""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2478,30,"""Bogotá D. C. (644 · 26.0%) · Antioquia (519 · 20.9%) · Cundinamarca (207 · 8.4%)"""
62,"""PROBLEMAS RELACIONADOS CON FACILIDADES DE ATENCIÓN MÉDICA U OTROS SERVICIOS DE SALUD""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS""","""Web""",2478,29,"""Bogotá D. C. (733 · 29.6%) · Antioquia (499 · 20.1%) · Valle del Cauca (226 · 9.1%)"""


codigo_departamento,departamento,perfiles_que_lidera,lidera_sin_empate,quejas_como_lider,pct_perfiles
str,str,u32,u32,u32,f64
"""11""","""Bogotá D. C.""",36045,27112,396225,31.67
"""05""","""Antioquia""",26092,15114,168280,22.93
"""76""","""Valle del Cauca""",10353,10251,34940,9.1
"""08""","""Atlántico""",6123,2981,12217,5.38
"""68""","""Santander""",4671,4175,19965,4.1
"""25""","""Cundinamarca""",4599,3425,7480,4.04
"""54""","""Norte de Santander""",2200,1798,3524,1.93
"""15""","""Boyacá""",2089,1262,2645,1.84
"""41""","""Huila""",1921,1354,3248,1.69


### 9.2 Ejercicio 2 · Cronología y dos perfiles concretos

Las ausencias de un perfil en un mes se muestran como **sin registros** (nulo), no
como cero. El indicador `por_mil_quejas_del_mes` relativiza cada serie frente al
volumen nacional del mes, para no confundir crecimiento general con tendencia del perfil.

In [18]:
# ── Ejercicio 2 · cronología, fechas inválidas y dos perfiles concretos (fuera del reloj) ──
cronologia = cpu_e2.filter(pl.col('anio').is_not_null() & pl.col('mes_num').is_not_null())
fechas_invalidas = cpu_e2.filter(pl.col('anio').is_null() | pl.col('mes_num').is_null())
assert int(cronologia['quejas'].sum()) + int(fechas_invalidas['quejas'].sum()) == TOTAL_ESPERADO
print(f'Reportes con fecha válida: {int(cronologia["quejas"].sum()):,} · '
      f'con fecha inválida (separados): {int(fechas_invalidas["quejas"].sum()):,}')

# Meses realmente observados (no se inventa calendario).
meses = (cronologia.group_by('anio', 'mes_num').agg(pl.col('quejas').sum().alias('quejas_mes'),
                                                    pl.len().alias('grupos_mes'))
         .sort('anio', 'mes_num'))
display(meses)

# Selección reproducible de dos perfiles (departamento + cinco dimensiones), con
# departamento y edad informados y presentes en todos los meses observados:
#   A = la serie con más quejas; B = la mayor con otro departamento y otro motivo.
series = (cronologia.drop_nulls(CLAVES_E1)
          .group_by(CLAVES_E1).agg(pl.col('quejas').sum().alias('total'),
                                   pl.len().alias('meses_con_quejas')))
completas = series.filter(pl.col('meses_con_quejas') == meses.height)
candidatas = (completas if completas.height >= 2 else series).sort(
    ['total', *CLAVES_E1], descending=[True] + [False] * len(CLAVES_E1))
perfil_a = candidatas.row(0, named=True)
perfil_b = candidatas.filter((pl.col('codigo_departamento') != perfil_a['codigo_departamento'])
                             & (pl.col('cod_motesp') != perfil_a['cod_motesp'])).row(0, named=True)

def serie_de(perfil: dict, etiqueta: str) -> pl.DataFrame:
    filtro = pl.all_horizontal([pl.col(c).eq_missing(perfil[c]) for c in CLAVES_E1])
    observada = cronologia.filter(filtro).select('anio', 'mes_num', 'quejas')
    return (meses.select('anio', 'mes_num', 'quejas_mes')
            .join(observada, on=['anio', 'mes_num'], how='left')
            .sort('anio', 'mes_num')
            .with_columns(
                pl.lit(etiqueta).alias('perfil'),
                pl.when(pl.col('quejas').is_null()).then(pl.lit('sin registros (no se imputa 0)'))
                  .otherwise(pl.lit('observado')).alias('estado'),
                pl.col('quejas').diff().alias('variacion_vs_mes_anterior'),
                (pl.col('quejas') / pl.col('quejas_mes') * 1000).round(2)
                  .alias('por_mil_quejas_del_mes')))

def resumen_serie(serie: pl.DataFrame) -> dict:
    obs = serie.drop_nulls('quejas')
    semestre = obs.with_columns(
        pl.format('{}-{}', 'anio', pl.when(pl.col('mes_num') <= 6).then(pl.lit('I'))
                  .otherwise(pl.lit('II'))).alias('semestre'))
    por_semestre = semestre.group_by('semestre').agg(pl.col('quejas').sum()).sort('semestre')
    pico = obs.sort('quejas', descending=True).row(0, named=True)
    valle = obs.sort('quejas').row(0, named=True)
    return {'meses_observados': obs.height,
            'total': int(obs['quejas'].sum()),
            'media_mensual': round(obs['quejas'].mean(), 1),
            'coef_variacion': round(obs['quejas'].std() / obs['quejas'].mean(), 3),
            'pico': f"{pico['anio']}-{pico['mes_num']:02d}: {pico['quejas']}",
            'valle': f"{valle['anio']}-{valle['mes_num']:02d}: {valle['quejas']}",
            'por_semestre': dict(zip(por_semestre['semestre'], por_semestre['quejas'])),
            'por_mil_primer_mes': obs['por_mil_quejas_del_mes'][0],
            'por_mil_ultimo_mes': obs['por_mil_quejas_del_mes'][-1]}

perfiles_seleccionados = {}
for etiqueta, perfil in (('A', perfil_a), ('B', perfil_b)):
    serie = serie_de(perfil, etiqueta)
    perfiles_seleccionados[etiqueta] = {
        'perfil': {c: perfil[c] for c in CLAVES_E1},
        'departamento': DEPARTAMENTOS.get(perfil['codigo_departamento']),
        'resumen': resumen_serie(serie), 'serie': serie}
    print(f'\nPerfil {etiqueta}:', json.dumps(perfiles_seleccionados[etiqueta]['perfil'],
                                             ensure_ascii=False))
    print(json.dumps(perfiles_seleccionados[etiqueta]['resumen'], ensure_ascii=False))
    with pl.Config(tbl_rows=30):
        display(serie.select('anio', 'mes_num', 'quejas', 'estado',
                             'variacion_vs_mes_anterior', 'quejas_mes', 'por_mil_quejas_del_mes'))

cpu_e2.write_parquet(OUT / 'e2_evolucion_mensual.parquet')
meses.write_csv(OUT / 'e2_totales_por_mes.csv')
fechas_invalidas.write_csv(OUT / 'e2_fechas_invalidas.csv')
pl.concat([v['serie'].with_columns(*[pl.lit(v['perfil'][c]).alias(c) for c in CLAVES_E1])
           for v in perfiles_seleccionados.values()], how='diagonal_relaxed'
          ).write_csv(OUT / 'e2_perfiles_seleccionados.csv')

Reportes con fecha válida: 2,444,766 · con fecha inválida (separados): 0


anio,mes_num,quejas_mes,grupos_mes
i16,i8,u32,u32
2023,7,116031,76103
2023,8,129984,81550
2023,9,128894,82645
2023,10,136090,85762
2023,11,122621,77916
…,…,…,…
2024,8,144558,83806
2024,9,159436,90865
2024,10,186192,100004



Perfil A: {"codigo_departamento": "11", "edad_anios": 0, "patologia_1": "MATERNO INFANTIL", "cod_motesp": "010201", "motivo_especifico": "NEGACIÓN EN LA ASIGNACIÓN DE CITAS O CONSULTAS", "pqr_canal": "Web"}
{"meses_observados": 18, "total": 886, "media_mensual": 49.2, "coef_variacion": 0.268, "pico": "2023-11: 71", "valle": "2024-12: 21", "por_semestre": {"2023-II": 315, "2024-I": 285, "2024-II": 286}, "por_mil_primer_mes": 0.24, "por_mil_ultimo_mes": 0.15}


anio,mes_num,quejas,estado,variacion_vs_mes_anterior,quejas_mes,por_mil_quejas_del_mes
i16,i8,u32,str,i64,u32,f64
2023,7,28,"""observado""",null,116031,0.24
2023,8,66,"""observado""",38,129984,0.51
2023,9,50,"""observado""",-16,128894,0.39
2023,10,58,"""observado""",8,136090,0.43
2023,11,71,"""observado""",13,122621,0.58
2023,12,42,"""observado""",-29,83077,0.51
2024,1,47,"""observado""",5,110994,0.42
2024,2,48,"""observado""",1,120943,0.4
2024,3,48,"""observado""",0,119787,0.4



Perfil B: {"codigo_departamento": "05", "edad_anios": 3, "patologia_1": "MATERNO INFANTIL", "cod_motesp": "010101", "motivo_especifico": "FALTA DE OPORTUNIDAD EN LAS CITAS O CONSULTAS", "pqr_canal": "Web"}
{"meses_observados": 18, "total": 540, "media_mensual": 30.0, "coef_variacion": 0.532, "pico": "2024-10: 64", "valle": "2023-12: 8", "por_semestre": {"2023-II": 113, "2024-I": 162, "2024-II": 265}, "por_mil_primer_mes": 0.31, "por_mil_ultimo_mes": 0.15}


anio,mes_num,quejas,estado,variacion_vs_mes_anterior,quejas_mes,por_mil_quejas_del_mes
i16,i8,u32,str,i64,u32,f64
2023,7,36,"""observado""",null,116031,0.31
2023,8,9,"""observado""",-27,129984,0.07
2023,9,16,"""observado""",7,128894,0.12
2023,10,25,"""observado""",9,136090,0.18
2023,11,19,"""observado""",-6,122621,0.15
2023,12,8,"""observado""",-11,83077,0.1
2024,1,13,"""observado""",5,110994,0.12
2024,2,26,"""observado""",13,120943,0.21
2024,3,22,"""observado""",-4,119787,0.18


### 9.3 Lectura de tiempos y control de entrega

In [19]:
# ── Lectura de los tiempos medidos: primera ejecución frente a mediana ──
UMBRAL_EMPATE = 0.05  # ±5 %: diferencias menores no se consideran concluyentes

def veredicto(factor: float) -> str:
    if factor > 1 + UMBRAL_EMPATE:
        return f'GPU más rápida (×{factor:.2f})'
    if factor < 1 - UMBRAL_EMPATE:
        return f'CPU más rápida (GPU tarda ×{1 / factor:.2f})'
    return f'empate práctico (factor {factor:.2f})'

primeras_df = pl.DataFrame(primeras)
lectura_tiempos = []
for nombre, corto in (('e1_departamentos', 'e1'), ('e2_evolucion_mensual', 'e2')):
    p = {m: primeras_df.filter((pl.col('ejercicio') == corto) & (pl.col('motor') == m))['segundos'].item()
         for m in ('cpu', 'gpu')}
    r = {m: resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == m)).row(0, named=True)
         for m in ('cpu', 'gpu')}
    factor_primera = p['cpu'] / p['gpu']
    factor_mediana = r['cpu']['mediana_s'] / r['gpu']['mediana_s']
    fila = {'ejercicio': nombre, 'grupos': controles[nombre]['grupos'],
            'primera_cpu_s': round(p['cpu'], 3), 'primera_gpu_s': round(p['gpu'], 3),
            'factor_primera': round(factor_primera, 3),
            'mediana_cpu_s': round(r['cpu']['mediana_s'], 3),
            'mediana_gpu_s': round(r['gpu']['mediana_s'], 3),
            'factor_mediana': round(factor_mediana, 3),
            'dispersion_cpu_pct': round((r['cpu']['max_s'] - r['cpu']['min_s']) / r['cpu']['mediana_s'] * 100, 1),
            'dispersion_gpu_pct': round((r['gpu']['max_s'] - r['gpu']['min_s']) / r['gpu']['mediana_s'] * 100, 1),
            'veredicto_primera': veredicto(factor_primera),
            'veredicto_mediana': veredicto(factor_mediana)}
    fila['conclusion_cambia'] = fila['veredicto_primera'].split(' (')[0] != fila['veredicto_mediana'].split(' (')[0]
    fila['costo_inicial_gpu_s'] = round(p['gpu'] - r['gpu']['mediana_s'], 3)
    lectura_tiempos.append(fila)
    print(f"{nombre}: {fila['grupos']:,} grupos · primera → {fila['veredicto_primera']} · "
          f"mediana → {fila['veredicto_mediana']} · ¿cambia la conclusión? "
          f"{'sí' if fila['conclusion_cambia'] else 'no'} · sobrecosto 1.ª GPU "
          f"{fila['costo_inicial_gpu_s']} s")
lectura_tiempos = pl.DataFrame(lectura_tiempos)
display(lectura_tiempos)
lectura_tiempos.write_csv(OUT / 'lectura_tiempos_PD06.csv')

e1_departamentos: 571,235 grupos · primera → CPU más rápida (GPU tarda ×4.56) · mediana → CPU más rápida (GPU tarda ×1.33) · ¿cambia la conclusión? no · sobrecosto 1.ª GPU 2.434 s
e2_evolucion_mensual: 1,492,458 grupos · primera → CPU más rápida (GPU tarda ×1.83) · mediana → CPU más rápida (GPU tarda ×1.56) · ¿cambia la conclusión? no · sobrecosto 1.ª GPU 1.536 s


ejercicio,grupos,primera_cpu_s,primera_gpu_s,factor_primera,mediana_cpu_s,mediana_gpu_s,factor_mediana,dispersion_cpu_pct,dispersion_gpu_pct,veredicto_primera,veredicto_mediana,conclusion_cambia,costo_inicial_gpu_s
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,bool,f64
"""e1_departamentos""",571235,0.694,3.168,0.219,0.551,0.734,0.751,3.8,30.1,"""CPU más rápida (GPU tarda ×4.5…","""CPU más rápida (GPU tarda ×1.3…",false,2.434
"""e2_evolucion_mensual""",1492458,1.632,2.991,0.545,0.936,1.456,0.643,63.6,34.5,"""CPU más rápida (GPU tarda ×1.8…","""CPU más rápida (GPU tarda ×1.5…",false,1.536


In [20]:
# ── Control JSON de entrega ──
from datetime import datetime

departamentos_observados = sorted(cpu_e1['codigo_departamento'].drop_nulls().unique().to_list())
control_pd06 = {
    'taller': 'PD06 · PQRS completas: agregaciones de alta cardinalidad en Polars CPU y GPU',
    'generado': datetime.now().astimezone().isoformat(timespec='seconds'),
    'entradas': [{'corte': periodo, 'csv': csv.name, 'separador': SEPARADORES[periodo],
                  'bytes_csv': csv.stat().st_size, 'filas_esperadas': esperado,
                  'filas_parquet': obtenido}
                 for periodo, csv, esperado, obtenido
                 in zip(PERIODOS, ARCHIVOS_CSV, FILAS_ESPERADAS, filas_por_archivo)],
    'total_reportes': int(sum(filas_por_archivo)),
    'total_esperado': TOTAL_ESPERADO,
    'total_conciliado': int(sum(filas_por_archivo)) == TOTAL_ESPERADO
                        and all(c['reportes'] == TOTAL_ESPERADO for c in controles.values()),
    'conversion': conversion.to_dicts(),
    'calidad_por_corte': calidad.to_dicts(),
    'reglas': {
        'version': VERSION_REGLAS,
        'edad': (f'afec_edad → Float64 (no numérica = nulo); válida si es finita, >= 0 y '
                 f'<= {EDAD_MAXIMA}; se trunca a años cumplidos (Int16). Vacía, negativa, '
                 'no numérica, NaN/inf o fuera de rango → edad_anios nula; el reporte se conserva.'),
        'fecha': (f'anio desde periodo y mes_num desde mes (número o nombre en español); '
                  f'válida si {ANIO_MINIMO} <= anio <= {ANIO_MAXIMO} y 1 <= mes <= 12; si no, '
                  'ambos nulos. Se cuentan en el total y se separan al presentar la cronología. '
                  'fecha_en_corte controla la coherencia con el semestre del archivo.'),
        'departamento': ('afec_cod_depto sin espacios, 1-2 dígitos, relleno a 2 (zfill) y '
                         'validado contra 33 códigos DIVIPOLA; si no, codigo_departamento nulo '
                         'y departamento_valido = False.'),
        'cod_motesp': (f'relleno a {ANCHO_COD_MOTESP} dígitos (2024_I omite ceros a la izquierda); '
                       'original en cod_motesp_origen; 1 código = 1 descripción verificado.'),
    },
    'cobertura_departamentos': {
        'esperados': len(DEPARTAMENTOS), 'observados': len(departamentos_observados),
        'codigos_observados': departamentos_observados,
        'faltantes': sorted(set(DEPARTAMENTOS) - set(departamentos_observados)),
        'reportes_sin_departamento_valido':
            controles['e1_departamentos']['nulos_por_clave']['codigo_departamento']['reportes']},
    'grupos_por_consulta': {n: c['grupos'] for n, c in controles.items()},
    'perfiles_distintos_e1': n_perfiles,
    'meses_observados_e2': [f'{a}-{m:02d}' for a, m in meses.select('anio', 'mes_num').iter_rows()],
    'reportes_fecha_invalida_e2': int(fechas_invalidas['quejas'].sum()),
    'controles_resultados': controles,
    'perfiles_seleccionados_e2': {k: {'perfil': v['perfil'], 'departamento': v['departamento'],
                                      'resumen': v['resumen']}
                                  for k, v in perfiles_seleccionados.items()},
    'entorno': entorno,
    'medicion': {
        'alcance': ('perf_counter alrededor de collect: leer los 3 Parquet + proyectar + '
                    'group_by/len + sort + materializar un DataFrame Polars en memoria CPU. '
                    'En GPU incluye transferencias host↔device y la conversión del resultado.'),
        'excluye': ['construcción del plan', 'descarga', 'conversión CSV→Parquet',
                    'verificaciones', 'comparación de resultados', 'exportación'],
        'motor_cpu': entorno['motor_cpu'], 'motor_gpu': entorno['motor_gpu'],
        'calentamiento': 'una vuelta CPU y GPU por ejercicio, excluida de las medianas',
        'repeticiones': REPETICIONES, 'alternancia': 'impar CPU→GPU, par GPU→CPU',
        'cache_sistema_operativo': ('no se vacía; tras la primera lectura los Parquet suelen '
                                    'quedar en la caché de páginas y ambos motores leen desde RAM'),
        'fallback_cpu_en_gpu': 'deshabilitado (raise_on_fail=True)'},
    'tiempos': {'primeras_ejecuciones': primeras, 'calentamiento': calentamiento,
                'resumen': resumen.to_dicts(), 'factores': factores,
                'lectura': lectura_tiempos.to_dicts()},
}
ruta_control = OUT / 'control_PD06.json'
ruta_control.write_text(json.dumps(control_pd06, indent=2, ensure_ascii=False, default=str),
                        encoding='utf-8')
print('Control JSON:', ruta_control)
print('Archivos en', OUT)
for ruta in sorted(OUT.glob('*.*')):
    print(f'  {ruta.name:45s} {ruta.stat().st_size / 2**20:8.2f} MB')

Control JSON: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06/control_PD06.json
Archivos en /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06
  benchmark_PD06.csv                                0.00 MB
  calentamiento_PD06.csv                            0.00 MB
  calidad_por_corte_PD06.csv                        0.00 MB
  control_PD06.json                                 0.01 MB
  conversion_PD06.csv                               0.00 MB
  e1_detalle_departamento_perfil.parquet            1.13 MB
  e1_liderazgo_departamentos.csv                    0.00 MB
  e1_perfiles_principales_top3.csv                  0.00 MB
  e1_ranking_top10_por_perfil.parquet               1.76 MB
  e2_evolucion_mensual.parquet                      2.67 MB
  e2_fechas_invalidas.csv                           0.00 MB
  e2_perfiles_seleccionados.csv                     0.00 MB
  e2_totales_por_mes.csv                            0.00 MB
  factores_PD06.csv   

## 10. Conclusiones

**Unidad de análisis.** Todas las cifras son **reportes de PQRD (quejas)**, no personas únicas
(un mismo afectado puede registrar varias) ni tasas por población. Se procesaron los tres cortes
completos: **2.444.766 reportes**, conciliados por corte y en cada consulta.

### 10.1 Preparación y calidad de datos

- Los tres CSV traen 38 columnas idénticas, pero con separadores distintos (`;`, `,`, `;`).
  **2024-I omite los ceros a la izquierda**: 167.477 reportes con `afec_cod_depto` de un dígito
  (`5` en lugar de `05`) y el 100 % de `cod_motesp` con cinco dígitos (`10304` en lugar de `010304`).
  Sin la normalización, un mismo departamento o motivo se habría partido en dos grupos y la
  comparación entre cortes no sería válida. Se verificó que cada uno de los 37 códigos de motivo
  específico tiene una sola descripción.
- Departamentos: 33/33 códigos DIVIPOLA observados; 0 reportes inválidos.
- Edad: 1 reporte con 121 años (fuera de la regla 0–120) queda en el grupo de edad nula y se
  conserva en el total. Fechas: 0 inválidas y 0 fuera del semestre de su archivo; 18 meses observados
  (julio 2023 a diciembre 2024).

### 10.2 Ejercicio 1 · ¿Qué departamentos lideran dentro de cada perfil?

- **Cardinalidad:** 571.235 combinaciones departamento × perfil y 113.803 perfiles distintos
  (edad en años × patología × motivo específico × canal).
- **Bogotá D. C. y Antioquia dominan los perfiles de mayor volumen.** En los 15 perfiles con más quejas,
  Bogotá ocupa el primer lugar en todos (26–31 % del perfil), Antioquia el segundo (12–22 %) y el tercero
  se alterna entre Valle del Cauca y Cundinamarca (8–11 %). Bogotá concentra el 21,4 % del total
  nacional de quejas, pero lidera el 31,7 % de los perfiles.
- **Patrón de los perfiles principales:** predomina el motivo *negación en la asignación de citas o
  consultas* (010201), radicado por **Web**, en dos poblaciones: primera infancia (0–4 años,
  *materno infantil*) y adultos de 56 a 68 años (*problemas relacionados con facilidades de atención
  médica*). El perfil de mayor volumen es 0 años · materno infantil · 010201 · Web, con 3.091 quejas en 31
  departamentos.
- **Cola larga y empates:** la mediana es de 3 quejas por perfil; 38.020 perfiles tienen una sola queja,
  41.101 aparecen en un único departamento y solo 4.621 superan las 100. En 31.616 perfiles el primer
  lugar está empatado, y el desempate reproducible por código ascendente favorece a códigos bajos como
  Antioquia (`05`). Por eso el notebook reporta `lidera_sin_empate`: con ese criterio, Antioquia pasa de
  26.092 a 15.114 perfiles liderados y Bogotá de 36.045 a 27.112. El top por perfil es concluyente en los
  perfiles con volumen, no en la cola.

### 10.3 Ejercicio 2 · ¿Cómo evolucionan los dos perfiles seleccionados?

El volumen nacional crece de forma clara: de 116.031 quejas en julio de 2023 a 186.192 en octubre de 2024.
Diciembre de 2023 cae a 83.077. Por eso cada perfil se lee también en **quejas por mil del mes**.

- **Perfil A · Bogotá · 0 años · materno infantil · 010201 negación en la asignación de citas · Web**
  (886 quejas, 18/18 meses). En términos absolutos es estable: 315, 285 y 286 por semestre, con una media
  de 49 al mes y un coeficiente de variación de 0,27. Su peso relativo, en cambio, **disminuye**: pasa de
  una media de 0,44 ‰ en 2023-II a 0,33 ‰ en 2024. El perfil no empeora; crece menos que el sistema. Noviembre y
  diciembre de 2024 (38 y 21) son los valores más bajos de la serie. Con dos puntos y un posible efecto
  de cierre de año, no se concluye tendencia.
- **Perfil B · Antioquia · 3 años · materno infantil · 010101 falta de oportunidad en citas · Web**
  (540 quejas, 18/18 meses). Crece en términos absolutos (113 → 162 → 265 por semestre) y también en
  términos relativos: media de 0,16 ‰ en 2023-II, con un pico aislado de 0,31 ‰ en julio de 2023,
  frente a 0,26–0,34 ‰ entre julio y octubre de 2024. El
  aumento supera al del volumen nacional y es una **señal a vigilar**. Sin embargo, la serie es muy
  volátil (CV 0,53; por ejemplo, 36 quejas en julio de 2023 frente a 9 en agosto, o 17 en mayo de 2024
  frente a 41 en junio), los conteos son pequeños y hay
  solo 18 meses: no basta para afirmar una tendencia estructural sin una prueba formal o más periodos.
- No se imputaron ceros: ambos perfiles tienen registros en los 18 meses. Si faltara alguno, el notebook
  lo marca como *sin registros*.

### 10.4 ¿Cuántos grupos genera cada consulta?

| Consulta | Grupos | Claves |
|---|---:|---|
| E1 · departamento × perfil | 571.235 | 6 (1 Int16 + 5 texto) |
| E2 · año × mes × departamento × perfil | 1.492.458 | 8 (Int16, Int8, Int16 + 5 texto) |

E2 genera unas 2,6 veces más grupos sobre la misma entrada: cada fila de E1 se reparte en 2,6 meses en
promedio. La salida pasa de ser cerca del 23 % de la entrada a cerca del 61 %, por lo que la agregación
apenas comprime.

### 10.5 Tiempos CPU frente a GPU · completar con la salida de 9.3 en tu equipo

> Las cifras deben copiarse de `lectura_tiempos_PD06.csv` después de ejecutar en WSL con CUDA.
> No se registran aquí valores no medidos.

| Consulta | 1.ª CPU (s) | 1.ª GPU (s) | Mediana CPU (s) | Mediana GPU (s) | Factor CPU/GPU | Veredicto |
|---|---:|---:|---:|---:|---:|---|
| E1 | … | … | … | … | … | … |
| E2 | … | … | … | … | … | … |

**Criterios para la lectura** (contrastar con las cifras medidas):

1. **¿La GPU acelera ambas?** Se responde con el factor de mediana. Mayor que 1,05 favorece a GPU,
   menor que 0,95 favorece a CPU y entre ambos hay empate práctico. La entrada es pequeña para una GPU:
   unos 57 MB de Parquet Zstandard y 2,4 millones de filas. Los costos fijos pesan proporcionalmente
   mucho: traducción del plan a cudf-polars, asignación de memoria con RMM, transferencia host→device y
   copia del resultado a CPU. Un factor cercano o inferior a 1 es un resultado válido.
2. **Lectura:** la decodificación Parquet y Zstandard, y la transferencia de 5 u 8 columnas
   (mayoritariamente texto) a VRAM, ocurren en cada `collect`. La caché de páginas del sistema operativo
   no se vacía, de modo que tras la primera lectura ambos motores leen desde RAM. Esto favorece por
   igual a CPU y GPU, pero la primera ejecución de cada motor puede incluir lectura desde disco.
3. **Transferencias y materialización:** E2 devuelve 1,49 millones de filas con 5 columnas de texto.
   La copia device→host y la construcción del DataFrame en CPU crecen con el tamaño de la salida. Si el
   factor de E2 es peor que el de E1, la causa más probable es este costo, no la agregación en sí.
4. **Ordenamiento:** ambas consultas ordenan por 7–8 claves con varias columnas de texto. La comparación
   de cadenas en la GPU es menos eficiente que la de enteros, y el orden debe ser total (`nulls_last`).
5. **Memoria:** verificar en `control_PD06.json` la VRAM total y la usada al registrar. Con este volumen
   no se espera saturación; si apareciera un error de memoria, se informa y no se mide como fallback.
6. **Primera ejecución frente a mediana:** la primera llamada GPU suele incluir la creación del contexto
   CUDA, la compilación o carga de kernels y el pool de RMM (`costo_inicial_gpu_s`). Si
   `conclusion_cambia` es *sí*, la conclusión correcta es la de la mediana, y la primera ejecución
   documenta el costo de arranque que pagaría un proceso que solo se ejecuta una vez.